In [14]:
content = """V[h_,s_,T_]:=(mH2*h^2)/2+(mS2*s^2)/2+(lamH*h^4)/4+(lamS*s^4)/4+(lamHS*h^2*s^2)/2
"""
with open("input.m", "w", encoding="utf-8") as f:
    f.write(content)

In [15]:
#!/usr/bin/env python3
"""
Convert a Mathematica scalar potential

    V[phi_, T_] := expr
or
    V[h_, s_, T_] := expr
or
    V[{h_, s_}, T_] := expr

into a Python module with

    V(X, T)
    gradV(X, T)

and, in the one-field case only,

    dV(phi, T)
"""

from pathlib import Path
import re
import sympy as sp
from sympy.parsing.mathematica import parse_mathematica

INPUT_FILE = "input.m"
OUTPUT_FILE = "potential_nd.py"


# ------------------------------------------------------------------
# Mathematica symbol normalization
# ------------------------------------------------------------------

MMA_SYMBOL_MAP = {
    "ϕ": "phi",
    "φ": "phi",
    "\\[CurlyPhi]": "phi",
    "\\[Phi]": "phi",
    "λ": "lam",
    "\\[Lambda]": "lam",
    "\\[Lambda]1H": "lam1H",
    "α": "alpha",
    "β": "beta",
    "γ": "gamma",
    "\\[Alpha]": "alpha",
    "\\[Beta]": "beta",
    "\\[Gamma]": "gamma",
    "μ": "mu",
    "\\[Mu]": "mu",
    "μ3US": "mu3US",
    "\\[Mu]3US": "mu3US",
    "Pi": "pi",
    "E": "E",
}


def replace_symbols(text: str) -> str:
    for old, new in sorted(MMA_SYMBOL_MAP.items(), key=lambda kv: -len(kv[0])):
        text = text.replace(old, new)
    return text


# ------------------------------------------------------------------
# Parsing utilities
# ------------------------------------------------------------------

import re

def remove_mathematica_comments(text: str) -> str:
    return re.sub(r"\(\*.*?\*\)", "", text, flags=re.DOTALL)


def normalize_whitespace(text: str) -> str:
    """
    Collapse all whitespace runs to single spaces.
    This makes multiline Mathematica definitions easier to parse.
    """
    return re.sub(r"\s+", " ", text).strip()


def split_top_level_commas(s: str):
    parts = []
    depth_paren = depth_brack = depth_brace = 0
    start = 0

    for i, ch in enumerate(s):
        if ch == "(":
            depth_paren += 1
        elif ch == ")":
            depth_paren -= 1
        elif ch == "[":
            depth_brack += 1
        elif ch == "]":
            depth_brack -= 1
        elif ch == "{":
            depth_brace += 1
        elif ch == "}":
            depth_brace -= 1
        elif ch == "," and depth_paren == depth_brack == depth_brace == 0:
            parts.append(s[start:i].strip())
            start = i + 1

    parts.append(s[start:].strip())
    return parts


def find_matching_bracket(text: str, start_idx: int, open_ch="[", close_ch="]"):
    depth = 0
    for i in range(start_idx, len(text)):
        ch = text[i]
        if ch == open_ch:
            depth += 1
        elif ch == close_ch:
            depth -= 1
            if depth == 0:
                return i
    raise ValueError("No matching closing bracket found.")


def parse_v_definition(src: str):
    """
    Parse a Mathematica definition of the form

        V[phi_, T_] := ...
        V[h_, s_, T_] := ...
        V[{h_, s_}, T_] := ...

    Also supports '=' instead of ':='.
    """
    src = remove_mathematica_comments(src)
    src = normalize_whitespace(src)

    vpos = src.find("V")
    if vpos == -1:
        raise ValueError("Could not find function name 'V' in the file.")

    i = vpos + 1
    while i < len(src) and src[i].isspace():
        i += 1

    if i >= len(src) or src[i] != "[":
        raise ValueError("Found 'V' but not followed by '['.")

    lbrack = i
    rbrack = find_matching_bracket(src, lbrack, "[", "]")

    lhs_inside = src[lbrack + 1:rbrack].strip()

    rest = src[rbrack + 1:].lstrip()
    if rest.startswith(":="):
        rhs = rest[2:].strip()
    elif rest.startswith("="):
        rhs = rest[1:].strip()
    else:
        raise ValueError("Expected ':=' or '=' after V[...].")

    args = split_top_level_commas(lhs_inside)
    if len(args) < 2:
        raise ValueError("Expected at least one field and one temperature argument.")

    T_part = args[-1].replace("_", "").strip()
    field_parts = args[:-1]

    if len(field_parts) == 1 and field_parts[0].startswith("{") and field_parts[0].endswith("}"):
        inner = field_parts[0][1:-1].strip()
        field_names = [x.replace("_", "").strip() for x in split_top_level_commas(inner)]
    else:
        field_names = [x.replace("_", "").strip() for x in field_parts]

    if not field_names:
        raise ValueError("No field names detected in V definition.")

    return field_names, T_part, rhs


# ------------------------------------------------------------------
# Mathematica -> SymPy conversion
# ------------------------------------------------------------------

def rewrite_special_constructs(expr):
    def _rw(e):
        if not isinstance(e, sp.Basic):
            return e

        if e.is_Atom:
            return e

        args = tuple(_rw(a) for a in e.args)
        name = getattr(e.func, "__name__", str(e.func))

        if name == "If" and len(args) == 3:
            cond, a, b = args
            return sp.Piecewise((a, cond), (b, True))

        if name == "Which" and len(args) >= 2 and len(args) % 2 == 0:
            pieces = []
            for k in range(0, len(args), 2):
                pieces.append((args[k + 1], args[k]))
            return sp.Piecewise(*pieces)

        if name == "Boole" and len(args) == 1:
            return sp.Piecewise((sp.Integer(1), args[0]), (sp.Integer(0), True))

        if name in ("UnitStep", "HeavisideTheta") and len(args) == 1:
            x = args[0]
            return sp.Piecewise(
                (sp.Integer(0), x < 0),
                (sp.Rational(1, 2), sp.Eq(x, 0)),
                (sp.Integer(1), True),
            )

        try:
            return e.func(*args)
        except Exception:
            return e

    return _rw(expr)


def mma_to_sympy_expr(rhs: str):
    rhs = replace_symbols(rhs)
    expr = parse_mathematica(rhs)
    expr = rewrite_special_constructs(expr)

    try:
        expr = sp.simplify(expr.doit())
    except Exception:
        try:
            expr = expr.doit()
        except Exception:
            pass

    return expr


# ------------------------------------------------------------------
# Numerical safety transforms
# ------------------------------------------------------------------

def make_half_powers_safe(expr: sp.Expr) -> sp.Expr:
    def repl(e):
        if isinstance(e, sp.Pow) and isinstance(e.exp, sp.Rational) and e.exp.q == 2:
            base = e.base
            if getattr(getattr(base, "func", None), "__name__", "") == "Abs":
                return sp.Pow(base, e.exp)
            return sp.Pow(sp.Abs(base), e.exp)
        return e

    return expr.replace(
        lambda e: isinstance(e, sp.Pow) and isinstance(e.exp, sp.Rational) and e.exp.q == 2,
        repl,
    )


# ------------------------------------------------------------------
# SymPy -> NumPy code printer
# ------------------------------------------------------------------

def sympy_to_numpy_code(expr: sp.Expr) -> str:
    def emit(e):
        if e == sp.pi:
            return "np.pi"
        if e == sp.E:
            return "np.e"
        if e is sp.true:
            return "True"
        if e is sp.false:
            return "False"
        if isinstance(e, sp.Integer):
            return str(int(e))
        if isinstance(e, sp.Float):
            return repr(float(e))
        if isinstance(e, sp.Rational):
            return f"({e.p}/{e.q})"

        if isinstance(e, sp.Symbol):
            return e.name

        if isinstance(e, sp.Add):
            return "(" + " + ".join(emit(a) for a in e.args) + ")"

        if isinstance(e, sp.Mul):
            coeff, rest = e.as_coeff_Mul()
            if coeff == -1:
                return f"(-{emit(rest)})"
            return "(" + " * ".join(emit(a) for a in e.args) + ")"

        if isinstance(e, sp.Pow):
            base, exp = e.args
            if exp == sp.Rational(1, 2):
                return f"_rtabs({emit(base)})"
            return f"({emit(base)} ** {emit(exp)})"

        if isinstance(e, sp.Piecewise):
            pieces = list(e.args)
            code = "np.nan"
            for val, cond in reversed(pieces):
                if cond is True or cond == True:
                    code = emit(val)
                else:
                    code = f"np.where({emit(cond)}, {emit(val)}, {code})"
            return code

        if isinstance(e, sp.Equality):
            return f"({emit(e.lhs)} == {emit(e.rhs)})"
        if isinstance(e, sp.Unequality):
            return f"({emit(e.lhs)} != {emit(e.rhs)})"
        if isinstance(e, sp.StrictLessThan):
            return f"({emit(e.lhs)} < {emit(e.rhs)})"
        if isinstance(e, sp.StrictGreaterThan):
            return f"({emit(e.lhs)} > {emit(e.rhs)})"
        if isinstance(e, sp.LessThan):
            return f"({emit(e.lhs)} <= {emit(e.rhs)})"
        if isinstance(e, sp.GreaterThan):
            return f"({emit(e.lhs)} >= {emit(e.rhs)})"

        if isinstance(e, sp.And):
            return "(" + " & ".join(emit(a) for a in e.args) + ")"
        if isinstance(e, sp.Or):
            return "(" + " | ".join(emit(a) for a in e.args) + ")"
        if isinstance(e, sp.Not):
            return f"(~{emit(e.args[0])})"

        fname = getattr(getattr(e, "func", None), "__name__", "")

        if fname == "Abs":
            return f"np.abs({emit(e.args[0])})"
        if fname == "log":
            return f"_logsafe({emit(e.args[0])})"
        if fname == "exp":
            return f"np.exp({emit(e.args[0])})"
        if fname == "sin":
            return f"np.sin({emit(e.args[0])})"
        if fname == "cos":
            return f"np.cos({emit(e.args[0])})"
        if fname == "tan":
            return f"np.tan({emit(e.args[0])})"
        if fname == "asin":
            return f"np.arcsin({emit(e.args[0])})"
        if fname == "acos":
            return f"np.arccos({emit(e.args[0])})"
        if fname == "atan":
            return f"np.arctan({emit(e.args[0])})"
        if fname == "sinh":
            return f"np.sinh({emit(e.args[0])})"
        if fname == "cosh":
            return f"np.cosh({emit(e.args[0])})"
        if fname == "tanh":
            return f"np.tanh({emit(e.args[0])})"
        if fname == "re":
            return f"np.real({emit(e.args[0])})"
        if fname == "im":
            return f"np.imag({emit(e.args[0])})"
        if fname == "Max":
            return "np.maximum.reduce([" + ", ".join(emit(a) for a in e.args) + "])"
        if fname == "Min":
            return "np.minimum.reduce([" + ", ".join(emit(a) for a in e.args) + "])"

        raise TypeError(f"Unsupported SymPy object: {type(e)} : {e}")

    return emit(expr)


# ------------------------------------------------------------------
# Main generator
# ------------------------------------------------------------------

def main():
    src = Path(INPUT_FILE).read_text(encoding="utf-8")

    field_names_raw, T_name_raw, rhs_raw = parse_v_definition(src)
    n_fields = len(field_names_raw)

    print("Detected fields:", field_names_raw)
    print("Detected T:", T_name_raw)

    X_syms = sp.symbols(" ".join(f"x{i}" for i in range(n_fields)), real=True)
    if n_fields == 1:
        X_syms = (X_syms,)
    T = sp.Symbol("T", real=True)

    expr = mma_to_sympy_expr(rhs_raw)

    subs_map = {sp.Symbol(T_name_raw): T}
    for i, nm in enumerate(field_names_raw):
        subs_map[sp.Symbol(nm)] = X_syms[i]
    expr = expr.xreplace(subs_map)

    expr = make_half_powers_safe(expr)

    free_syms = sorted(expr.free_symbols, key=lambda s: s.name)
    reserved_names = {T.name} | {x.name for x in X_syms}
    param_syms = [s for s in free_syms if s.name not in reserved_names]
    param_names = [s.name for s in param_syms]
    param_names_str = ", ".join(param_names) if param_names else "(none)"

    V_numpy = sympy_to_numpy_code(expr)

    if n_fields == 1:
        field_unpack = "    x0 = X"
    else:
        field_unpack = "\n".join([f"    x{i} = X[..., {i}]" for i in range(n_fields)])

    if param_names:
        param_extract = "\n".join([f"    {p} = params['{p}']" for p in param_names])
    else:
        param_extract = "    # no extra parameters"

    if len(field_names_raw) == 1:
        original_signature = f"V[{field_names_raw[0]}_, {T_name_raw}_]"
    else:
        original_signature = f"V[{', '.join(f + '_' for f in field_names_raw)}, {T_name_raw}_]"

    if n_fields == 1:
        prepare_x_code = """def _prepare_X(X):
    return np.asarray(X, dtype=float)
"""
        grad_body = """def gradV(X, T, h_rel=1e-3, h_abs=1e-5):
    T = float(T)
    X = _prepare_X(X).astype(float)

    h = h_rel * np.maximum(np.abs(X), 1.0) + h_abs
    return (V(X + h, T) - V(X - h, T)) / (2.0 * h)
"""
        one_field_alias = """
def dV(phi, T, h_rel=1e-3, h_abs=1e-5):
    phi = np.asarray(phi, dtype=float)
    h = h_rel * np.maximum(np.abs(phi), 1.0) + h_abs
    return (V(phi + h, T) - V(phi - h, T)) / (2.0 * h)
"""
    else:
        prepare_x_code = f"""def _prepare_X(X):
    X = np.asarray(X, dtype=float)
    if X.shape[-1] != {n_fields}:
        raise ValueError(f"Expected X.shape[-1] == {n_fields}, got {{X.shape}}")
    return X
"""
        grad_body = """def gradV(X, T, h_rel=1e-3, h_abs=1e-5):
    T = float(T)
    X = _prepare_X(X).astype(float)

    g = np.zeros_like(X, dtype=float)
    for i in range(X.shape[-1]):
        xi = X[..., i]
        hi = h_rel * np.maximum(np.abs(xi), 1.0) + h_abs

        dX = np.zeros_like(X, dtype=float)
        dX[..., i] = hi

        g[..., i] = (V(X + dX, T) - V(X - dX, T)) / (2.0 * hi)

    return g
"""
        one_field_alias = ""

    out = f'''"""
Auto-generated from Mathematica potential.

Original definition:
    {original_signature} := {rhs_raw}

Detected fields:
    {", ".join(field_names_raw)}

Detected free parameters (besides fields, T):
    {param_names_str}

You MUST implement get_params(T).
"""

import numpy as np


def _rtabs(x, floor=1e-30):
    x = np.asarray(x, dtype=float)
    return np.sqrt(np.maximum(np.abs(x), floor))

def _logsafe(x, floor=1e-30):
    x = np.asarray(x, dtype=float)
    return np.log(np.maximum(np.abs(x), floor))


def get_params(T):
    raise NotImplementedError(
        "Implement get_params(T) so that it returns a dict with keys: {param_names_str}"
    )


{prepare_x_code}

def V(X, T):
    T = float(T)
    X = _prepare_X(X)
{field_unpack}

    params = get_params(T)
{param_extract}

    out = {V_numpy}
    return np.asarray(out)


{grad_body}{one_field_alias}
'''

    Path(OUTPUT_FILE).write_text(out, encoding="utf-8")
    print(f"Wrote {OUTPUT_FILE}")
    print(f"Fields: {field_names_raw}")
    print(f"Parameters: {param_names_str}")


if __name__ == "__main__":
    main()

Detected fields: ['h', 's']
Detected T: T
Wrote potential_nd.py
Fields: ['h', 's']
Parameters: lamH, lamHS, lamS, mH2, mS2


In [16]:
with open("potential_nd.py") as f:
    print(f.read())

"""
Auto-generated from Mathematica potential.

Original definition:
    V[h_, s_, T_] := (mH2*h^2)/2+(mS2*s^2)/2+(lamH*h^4)/4+(lamS*s^4)/4+(lamHS*h^2*s^2)/2

Detected fields:
    h, s

Detected free parameters (besides fields, T):
    lamH, lamHS, lamS, mH2, mS2

You MUST implement get_params(T).
"""

import numpy as np


def _rtabs(x, floor=1e-30):
    x = np.asarray(x, dtype=float)
    return np.sqrt(np.maximum(np.abs(x), floor))

def _logsafe(x, floor=1e-30):
    x = np.asarray(x, dtype=float)
    return np.log(np.maximum(np.abs(x), floor))


def get_params(T):
    raise NotImplementedError(
        "Implement get_params(T) so that it returns a dict with keys: lamH, lamHS, lamS, mH2, mS2"
    )


def _prepare_X(X):
    X = np.asarray(X, dtype=float)
    if X.shape[-1] != 2:
        raise ValueError(f"Expected X.shape[-1] == 2, got {X.shape}")
    return X


def V(X, T):
    T = float(T)
    X = _prepare_X(X)
    x0 = X[..., 0]
    x1 = X[..., 1]

    params = get_params(T)
    lamH